# 深度域确定性子波提取

本 notebook 针对 `2-ANP-2A-RJS`：从 TVDSS 深度域地震提取井旁道，用井速构建局部相对时深表，并用稳定最小二乘反褶积提取 101 ms 确定性子波。


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from scipy.interpolate import interp1d
from scipy.signal import fftconvolve

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

src_root = repo_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

from cup.petrel.load import import_well_heads_petrel, load_vp_rho_logset_from_las
from cup.seismic.survey import open_survey
from wtie.optimize.wavelet import compute_expected_wavelet, get_spectrum
from wtie.processing import grid
from wtie.utils.datasets import tutorial

plt.rcParams["figure.dpi"] = 120
pd.set_option("display.max_columns", 30)

In [ ]:
well_name = "2-ANP-2A-RJS"

data_root = repo_root / "data"
las_file = data_root / "vertical_well_las_target_clear" / f"{well_name}.las"
well_heads_file = data_root / "raw" / "well_heads"
seismic_file = data_root / "raw" / "mero_84_coord_extend"
output_dir = data_root / "output_deterministic_wavelet_depth_tvdss_20260423"
output_dir.mkdir(parents=True, exist_ok=True)

vp_mnemonic = "DTCO_QYZ_CLAER"
rho_mnemonic = "RHOZ_QYZ_CLAER"
target_wavelet_ms = 101.0

tutorial_folder = data_root / "tutorial"
model_state_dict = tutorial_folder / "trained_net_state_dict.pt"
network_parameters_file = tutorial_folder / "network_parameters.yaml"

segy_options = {
    "iline": 5,
    "xline": 21,
    "istep": 1,
    "xstep": 4,
}

for path in (las_file, well_heads_file, seismic_file, model_state_dict, network_parameters_file):
    assert path.exists(), f"Missing input file: {path}"

print("LAS:", las_file)
print("Well heads:", well_heads_file)
print("Seismic:", seismic_file)
print("Output dir:", output_dir)

## 工具函数


In [ ]:
def _as_float_array(values) -> np.ndarray:
    return np.asarray(values, dtype=float).copy()


def robust_fill_nans(x: np.ndarray) -> np.ndarray:
    x = _as_float_array(x)
    idx = np.arange(x.size)
    valid = np.isfinite(x)
    if valid.sum() == 0:
        raise ValueError("Cannot fill an all-NaN array.")
    if valid.sum() == x.size:
        return x
    return np.interp(idx, idx[valid], x[valid])


def zscore_trace(x: np.ndarray) -> np.ndarray:
    x = robust_fill_nans(x)
    x = x - np.nanmean(x)
    scale = np.nanstd(x)
    if not np.isfinite(scale) or scale <= 0:
        raise ValueError("Trace has non-positive standard deviation.")
    return x / scale


def compute_reflectivity_from_ai(ai: np.ndarray) -> np.ndarray:
    ai = robust_fill_nans(ai)
    upper = ai[:-1]
    lower = ai[1:]
    denom = lower + upper
    refl = np.zeros_like(ai)
    valid = np.isfinite(denom) & (np.abs(denom) > 0)
    refl[1:][valid] = (lower[valid] - upper[valid]) / denom[valid]
    return refl


def make_local_tdt(depth_tvdss: np.ndarray, vp_mps: np.ndarray) -> pd.DataFrame:
    depth_tvdss = _as_float_array(depth_tvdss)
    vp_mps = robust_fill_nans(vp_mps)
    order = np.argsort(depth_tvdss)
    depth_tvdss = depth_tvdss[order]
    vp_mps = vp_mps[order]
    if depth_tvdss.size < 3:
        raise ValueError("Need at least 3 depth samples to build a local TDT.")
    if np.nanmin(vp_mps) <= 0:
        raise ValueError("Vp must be positive after cleaning.")

    dz = np.diff(depth_tvdss)
    if np.any(dz <= 0):
        raise ValueError("TVDSS depth samples must be strictly increasing.")
    interval_vp = 0.5 * (vp_mps[:-1] + vp_mps[1:])
    dtwt = 2.0 * dz / interval_vp
    twt = np.concatenate([[0.0], np.cumsum(dtwt)])
    return pd.DataFrame({"tvdss_m": depth_tvdss, "twt_s": twt, "vp_mps": vp_mps})


def depth_curve_to_twt(depth: np.ndarray, values: np.ndarray, tdt: pd.DataFrame, dt: float) -> tuple[np.ndarray, np.ndarray]:
    depth = _as_float_array(depth)
    values = robust_fill_nans(values)
    tdt_depth = tdt["tvdss_m"].to_numpy(dtype=float)
    tdt_twt = tdt["twt_s"].to_numpy(dtype=float)
    overlap_min = max(float(np.nanmin(depth)), float(tdt_depth[0]))
    overlap_max = min(float(np.nanmax(depth)), float(tdt_depth[-1]))
    if overlap_max <= overlap_min:
        raise ValueError("Depth curve and local TDT do not overlap.")

    sample_depth = np.arange(overlap_min, overlap_max + 0.5 * np.nanmedian(np.diff(tdt_depth)), np.nanmedian(np.diff(tdt_depth)))
    value_at_depth = np.interp(sample_depth, depth, values)
    twt_at_depth = np.interp(sample_depth, tdt_depth, tdt_twt)
    twt_regular = np.arange(twt_at_depth[0], twt_at_depth[-1] + 0.5 * dt, dt)
    interp = interp1d(twt_at_depth, value_at_depth, bounds_error=False, fill_value=(value_at_depth[0], value_at_depth[-1])) # type: ignore
    return twt_regular, interp(twt_regular)


def odd_sample_count(duration_s: float, dt_s: float) -> int:
    n = int(round(duration_s / dt_s))
    if n % 2 == 0:
        n += 1
    return max(n, 3)


def convolution_design_matrix(reflectivity: np.ndarray, n_wavelet: int) -> np.ndarray:
    r = np.asarray(reflectivity, dtype=float)
    n = r.size
    center = n_wavelet // 2
    padded = np.pad(r, (center, n_wavelet - center - 1), mode="constant")
    g = np.empty((n, n_wavelet), dtype=float)
    for j in range(n_wavelet):
        g[:, j] = padded[n_wavelet - 1 - j : n_wavelet - 1 - j + n]
    return g


def roughness_operator(n: int) -> np.ndarray:
    if n < 3:
        return np.eye(n)
    d = np.zeros((n - 2, n), dtype=float)
    for i in range(n - 2):
        d[i, i : i + 3] = [1.0, -2.0, 1.0]
    return d


def deterministic_wavelet_scan(seismic: np.ndarray, reflectivity: np.ndarray, dt: float, wavelet_ms: float) -> dict:
    s = zscore_trace(seismic)
    r = robust_fill_nans(reflectivity)
    r = r - np.mean(r)
    n_wavelet = odd_sample_count(wavelet_ms / 1000.0, dt)
    g = convolution_design_matrix(r, n_wavelet)
    d = roughness_operator(n_wavelet)
    gtg = g.T @ g
    gts = g.T @ s
    dtd = d.T @ d

    lambdas = np.r_[0.0, np.logspace(-8, 2, 41)]
    rows = []
    best = None
    for lam in lambdas:
        a = gtg + lam * dtd + 1e-10 * np.eye(n_wavelet)
        w = np.linalg.solve(a, gts)
        synth = g @ w
        corr = float(np.corrcoef(s, synth)[0, 1]) if np.std(synth) > 0 else np.nan
        nrmse = float(np.linalg.norm(s - synth) / np.linalg.norm(s))
        roughness = float(np.linalg.norm(d @ w) / max(np.linalg.norm(w), 1e-12))
        score = corr - 0.02 * np.log10(1.0 + roughness)
        row = {"lambda": float(lam), "corr": corr, "nrmse": nrmse, "roughness": roughness, "score": float(score)}
        rows.append(row)
        if best is None or row["score"] > best["metrics"]["score"]:
            best = {"wavelet": w, "synthetic": synth, "metrics": row}

    assert best is not None
    t = (np.arange(n_wavelet) - n_wavelet // 2) * dt
    best["wavelet_time_s"] = t
    best["scan"] = pd.DataFrame(rows)
    best["seismic_norm"] = s
    best["reflectivity_used"] = r
    return best


def savefig(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=180, bbox_inches="tight")
    print("Saved", path)

## 1) 读取井曲线和井头


In [ ]:
well_heads_df = import_well_heads_petrel(well_heads_file)
well_row = well_heads_df.loc[well_heads_df["Name"] == well_name]
assert not well_row.empty, f"well_name={well_name} not found in well_heads"
assert well_row.shape[0] == 1, f"well_name={well_name} duplicated in well_heads: {well_row.shape[0]}"

well_row = well_row.iloc[0]
kb_m = float(well_row["Well datum value"])
well_x = float(well_row["Surface X"])
well_y = float(well_row["Surface Y"])

logset_md = load_vp_rho_logset_from_las(
    las_file,
    vp_mnemonic=vp_mnemonic,
    rho_mnemonic=rho_mnemonic,
    vp_unit="us/m",
    rho_unit="g/cm3",
)

md_m = logset_md.basis.astype(float)
tvdss_m = md_m - kb_m
vp_mps = robust_fill_nans(logset_md.Vp.values)
rho_gcc = robust_fill_nans(logset_md.Rho.values)
ai = vp_mps * rho_gcc
reflectivity_depth = compute_reflectivity_from_ai(ai)

well_df = pd.DataFrame({
    "md_m": md_m,
    "tvdss_m": tvdss_m,
    "vp_mps": vp_mps,
    "rho_gcc": rho_gcc,
    "ai": ai,
    "reflectivity": reflectivity_depth,
})

print(f"Well XY=({well_x:.2f}, {well_y:.2f}), KB={kb_m:.2f} m")
print(well_df[["md_m", "tvdss_m", "vp_mps", "rho_gcc", "ai"]].describe())

## 2) 提取 TVDSS 深度域井旁道


In [ ]:
survey = open_survey(seismic_file, seismic_type="segy", segy_options=segy_options)
geometry_depth = survey.query_geometry(domain="depth")
print(json.dumps(geometry_depth, indent=2, ensure_ascii=False))

il_float, xl_float = survey.coord_to_line(well_x, well_y)
print(f"Well line position: inline={il_float:.3f}, xline={xl_float:.3f}")
assert geometry_depth["inline_min"] <= il_float <= geometry_depth["inline_max"]
assert geometry_depth["xline_min"] <= xl_float <= geometry_depth["xline_max"]

seismic_depth_trace = survey.import_seismic_at_well(
    well_x=well_x,
    well_y=well_y,
    domain="depth",
)

seis_depth = seismic_depth_trace.basis.astype(float)
seis_amp = robust_fill_nans(seismic_depth_trace.values)
print(f"Depth trace: z=[{seis_depth[0]:.2f}, {seis_depth[-1]:.2f}] m, dz={seismic_depth_trace.sampling_rate:.3f} m, n={seis_depth.size}")
assert seismic_depth_trace.is_md, "Depth-domain seismic is represented by grid.Seismic basis_type='md'."

## 3) 共同深度窗和局部时深表


In [ ]:
overlap_z_min = max(float(well_df["tvdss_m"].min()), float(seis_depth.min()))
overlap_z_max = min(float(well_df["tvdss_m"].max()), float(seis_depth.max()))
assert overlap_z_max > overlap_z_min, (
    f"No TVDSS overlap. well=[{well_df['tvdss_m'].min()}, {well_df['tvdss_m'].max()}], "
    f"seismic=[{seis_depth.min()}, {seis_depth.max()}]"
)

well_win = well_df.loc[(well_df["tvdss_m"] >= overlap_z_min) & (well_df["tvdss_m"] <= overlap_z_max)].copy()
assert len(well_win) > 10, f"Too few well samples in overlap window: {len(well_win)}"

local_tdt = make_local_tdt(well_win["tvdss_m"].to_numpy(), well_win["vp_mps"].to_numpy())
local_tdt_path = output_dir / f"local_tdt_{well_name}.csv"
local_tdt.to_csv(local_tdt_path, index=False)

print(f"Overlap TVDSS window: {overlap_z_min:.2f} - {overlap_z_max:.2f} m")
print(f"Local TWT window: {local_tdt['twt_s'].iloc[0]:.4f} - {local_tdt['twt_s'].iloc[-1]:.4f} s")
print("Saved", local_tdt_path)
local_tdt.head()

## 4) 转换到时间域并匹配窗口


In [ ]:
dt_from_depth = float(np.nanmedian(2.0 * np.diff(local_tdt["tvdss_m"].to_numpy()) / local_tdt["vp_mps"].to_numpy()[1:]))
dt_s = round(dt_from_depth, 4)
dt_s = max(dt_s, 0.001)
print(f"Estimated time sampling: raw={dt_from_depth:.6f} s, using dt={dt_s:.4f} s")

twt_seis, seis_twt = depth_curve_to_twt(seis_depth, seis_amp, local_tdt, dt_s)
twt_vp, vp_twt = depth_curve_to_twt(well_df["tvdss_m"].to_numpy(), vp_mps, local_tdt, dt_s)
twt_rho, rho_twt = depth_curve_to_twt(well_df["tvdss_m"].to_numpy(), rho_gcc, local_tdt, dt_s)
twt_ai, ai_twt = depth_curve_to_twt(well_df["tvdss_m"].to_numpy(), ai, local_tdt, dt_s)
twt_ref, ref_twt = depth_curve_to_twt(well_df["tvdss_m"].to_numpy(), reflectivity_depth, local_tdt, dt_s)

t_min = max(twt_seis[0], twt_ref[0])
t_max = min(twt_seis[-1], twt_ref[-1])
t_common = np.arange(t_min, t_max + 0.5 * dt_s, dt_s)
assert t_common.size > 10, f"Too few common TWT samples: {t_common.size}"

seis_match_raw = np.interp(t_common, twt_seis, seis_twt)
ref_match = np.interp(t_common, twt_ref, ref_twt)
vp_match = np.interp(t_common, twt_vp, vp_twt)
rho_match = np.interp(t_common, twt_rho, rho_twt)
ai_match = np.interp(t_common, twt_ai, ai_twt)

assert np.isfinite(seis_match_raw).any(), "Time-domain seismic is all NaN."
assert np.isfinite(ref_match).any(), "Time-domain reflectivity is all NaN."

matched_df = pd.DataFrame({
    "twt_s": t_common,
    "seismic_raw": seis_match_raw,
    "reflectivity": ref_match,
    "vp_mps": vp_match,
    "rho_gcc": rho_match,
    "ai": ai_match,
})
matched_path = output_dir / f"matched_trace_reflectivity_{well_name}.csv"
matched_df.to_csv(matched_path, index=False)
print("Saved", matched_path)
matched_df.head()

## 5) 确定性子波提取


In [ ]:
wavelet_result = deterministic_wavelet_scan(
    seismic=matched_df["seismic_raw"].to_numpy(),
    reflectivity=matched_df["reflectivity"].to_numpy(),
    dt=dt_s,
    wavelet_ms=target_wavelet_ms,
)

wavelet_t = wavelet_result["wavelet_time_s"]
wavelet_amp = wavelet_result["wavelet"]
synthetic = wavelet_result["synthetic"]
seismic_norm = wavelet_result["seismic_norm"]
metrics = wavelet_result["metrics"]
scan_df = wavelet_result["scan"]

assert wavelet_amp.size % 2 == 1, "Wavelet sample count must be odd."
actual_wavelet_ms = 1000.0 * wavelet_amp.size * dt_s
wavelet_span_ms = 1000.0 * (wavelet_t[-1] - wavelet_t[0])
print(f"Wavelet samples={wavelet_amp.size}, nominal length={actual_wavelet_ms:.1f} ms, span={wavelet_span_ms:.1f} ms, dt={dt_s * 1000:.1f} ms")
print(metrics)

wavelet_df = pd.DataFrame({"time_s": wavelet_t, "amplitude": wavelet_amp})
wavelet_path = output_dir / f"wavelet_{well_name}.csv"
wavelet_df.to_csv(wavelet_path, index=False)

qc_df = matched_df.copy()
qc_df["seismic_norm"] = seismic_norm
qc_df["synthetic"] = synthetic
qc_df["residual"] = seismic_norm - synthetic
qc_df.attrs.update(metrics)
qc_path = output_dir / f"synthetic_qc_{well_name}.csv"
qc_df.to_csv(qc_path, index=False)
scan_path = output_dir / f"wavelet_lambda_scan_{well_name}.csv"
scan_df.to_csv(scan_path, index=False)

summary_path = output_dir / f"run_summary_{well_name}.json"
summary = {
    "well_name": well_name,
    "kb_m": kb_m,
    "well_x": well_x,
    "well_y": well_y,
    "overlap_tvdss_min_m": overlap_z_min,
    "overlap_tvdss_max_m": overlap_z_max,
    "dt_s": dt_s,
    "target_wavelet_ms": target_wavelet_ms,
    "actual_wavelet_ms": actual_wavelet_ms,
    "wavelet_span_ms": wavelet_span_ms,
    "best_metrics": metrics,
}
summary_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")

print("Saved", wavelet_path)
print("Saved", qc_path)
print("Saved", scan_path)
print("Saved", summary_path)

## 6) `compute_expected_wavelet` 计算


In [ ]:
with open(network_parameters_file, "r", encoding="utf-8") as f:
    training_parameters = yaml.load(f, Loader=yaml.Loader)

wavelet_extractor = tutorial.load_wavelet_extractor(training_parameters, model_state_dict)
expected_dt = float(wavelet_extractor.expected_sampling)
print(f"compute_expected_wavelet expected sampling: {expected_dt:.4f} s")

twt_expected = np.arange(t_common[0], t_common[-1] + 0.5 * expected_dt, expected_dt)
seis_expected = np.interp(twt_expected, matched_df["twt_s"].to_numpy(), matched_df["seismic_raw"].to_numpy())
ref_expected = np.interp(twt_expected, matched_df["twt_s"].to_numpy(), matched_df["reflectivity"].to_numpy())

seismic_for_expected = grid.Seismic(seis_expected, twt_expected, "twt", name="Depth-derived seismic for expected wavelet")
reflectivity_for_expected = grid.Reflectivity(ref_expected, twt_expected, name="Depth-derived reflectivity for expected wavelet")

expected_wavelet = compute_expected_wavelet(
    evaluator=wavelet_extractor,
    seismic=seismic_for_expected,
    reflectivity=reflectivity_for_expected,
    n_sampling=20,
    inverse_polarity=False,
    zero_phasing=False,
)

expected_wavelet_df = pd.DataFrame({
    "time_s": expected_wavelet.basis,
    "amplitude": expected_wavelet.values,
})
expected_wavelet_path = output_dir / f"compute_expected_wavelet_{well_name}.csv"
expected_wavelet_df.to_csv(expected_wavelet_path, index=False)

expected_input_path = output_dir / f"compute_expected_wavelet_input_{well_name}.csv"
pd.DataFrame({
    "twt_s": twt_expected,
    "seismic_raw": seis_expected,
    "reflectivity": ref_expected,
}).to_csv(expected_input_path, index=False)

expected_seismic_norm = zscore_trace(seis_expected)
expected_ref_used = robust_fill_nans(ref_expected)
expected_ref_used = expected_ref_used - np.mean(expected_ref_used)
expected_g = convolution_design_matrix(expected_ref_used, expected_wavelet.size)
expected_synthetic_raw = expected_g @ expected_wavelet.values
expected_scale = float(np.dot(expected_seismic_norm, expected_synthetic_raw) / max(np.dot(expected_synthetic_raw, expected_synthetic_raw), 1e-12))
expected_synthetic = expected_scale * expected_synthetic_raw
expected_corr = float(np.corrcoef(expected_seismic_norm, expected_synthetic)[0, 1]) if np.std(expected_synthetic) > 0 else np.nan
expected_nrmse = float(np.linalg.norm(expected_seismic_norm - expected_synthetic) / np.linalg.norm(expected_seismic_norm))

expected_synthetic_qc_path = output_dir / f"compute_expected_wavelet_synthetic_qc_{well_name}.csv"
pd.DataFrame({
    "twt_s": twt_expected,
    "seismic_norm": expected_seismic_norm,
    "reflectivity": expected_ref_used,
    "synthetic": expected_synthetic,
    "residual": expected_seismic_norm - expected_synthetic,
}).to_csv(expected_synthetic_qc_path, index=False)

expected_metrics = {
    "expected_wavelet_dt_s": expected_dt,
    "expected_wavelet_samples": int(expected_wavelet.size),
    "expected_wavelet_synthetic_scale": expected_scale,
    "expected_wavelet_synthetic_corr": expected_corr,
    "expected_wavelet_synthetic_nrmse": expected_nrmse,
}
if summary_path.exists():
    summary_data = json.loads(summary_path.read_text(encoding="utf-8"))
else:
    summary_data = {}
summary_data.update(expected_metrics)
summary_path.write_text(json.dumps(summary_data, indent=2, ensure_ascii=False), encoding="utf-8")

print("Saved", expected_wavelet_path)
print("Saved", expected_input_path)
print("Saved", expected_synthetic_qc_path)
print(f"Expected wavelet samples={expected_wavelet.size}, dt={expected_wavelet.sampling_rate * 1000:.1f} ms")
print(f"Expected-wavelet synthetic: corr={expected_corr:.3f}, nrmse={expected_nrmse:.3f}, scale={expected_scale:.3g}")
expected_wavelet_df.head()

## 7) QC 图件


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 5), sharey=False)
axes[0].plot(well_df["vp_mps"], well_df["tvdss_m"], lw=0.8)
axes[0].axhspan(overlap_z_min, overlap_z_max, color="tab:green", alpha=0.15)
axes[0].invert_yaxis()
axes[0].set_xlabel("Vp (m/s)")
axes[0].set_ylabel("TVDSS (m)")
axes[0].set_title("Well Vp and overlap")
axes[0].grid(True, alpha=0.25)

axes[1].plot(seis_amp, seis_depth, lw=0.8, color="tab:gray")
axes[1].axhspan(overlap_z_min, overlap_z_max, color="tab:green", alpha=0.15)
axes[1].invert_yaxis()
axes[1].set_xlabel("Amplitude")
axes[1].set_title("Depth-domain well trace")
axes[1].grid(True, alpha=0.25)

axes[2].plot(local_tdt["twt_s"] * 1000.0, local_tdt["tvdss_m"], lw=1.1, color="tab:blue")
axes[2].invert_yaxis()
axes[2].set_xlabel("Relative TWT (ms)")
axes[2].set_title("Local time-depth table")
axes[2].grid(True, alpha=0.25)

fig_depth_time_path = output_dir / f"qc_01_depth_time_match_{well_name}.png"
savefig(fig_depth_time_path)
plt.show()

In [ ]:
freq_hz = np.fft.rfftfreq(wavelet_amp.size, d=dt_s)
amp_spec = np.abs(np.fft.rfft(wavelet_amp))
phase_deg = np.unwrap(np.angle(np.fft.rfft(wavelet_amp))) * 180.0 / np.pi

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
axes[0].plot(wavelet_t * 1000.0, wavelet_amp, lw=1.3, color="tab:blue")
axes[0].axvline(0.0, color="black", lw=0.8, alpha=0.5)
axes[0].set_xlabel("Time (ms)")
axes[0].set_title("Deterministic LS wavelet")
axes[0].grid(True, alpha=0.25)

axes[1].plot(freq_hz, amp_spec, lw=1.2, color="tab:orange")
axes[1].set_xlim(0, min(125, freq_hz[-1]))
axes[1].set_xlabel("Frequency (Hz)")
axes[1].set_title("Amplitude spectrum")
axes[1].grid(True, alpha=0.25)

axes[2].plot(freq_hz, phase_deg, lw=1.0, color="tab:green")
axes[2].set_xlim(0, min(125, freq_hz[-1]))
axes[2].set_xlabel("Frequency (Hz)")
axes[2].set_ylabel("Phase (deg)")
axes[2].set_title("Unwrapped phase")
axes[2].grid(True, alpha=0.25)

fig_det_wavelet_path = output_dir / f"qc_02_deterministic_wavelet_spectrum_{well_name}.png"
savefig(fig_det_wavelet_path)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 5), sharey=True)
t_ms = matched_df["twt_s"].to_numpy() * 1000.0

axes[0].plot(matched_df["reflectivity"], t_ms, lw=0.8, color="tab:purple")
axes[0].invert_yaxis()
axes[0].set_xlabel("Reflectivity")
axes[0].set_ylabel("Relative TWT (ms)")
axes[0].set_title("Reflectivity")
axes[0].grid(True, alpha=0.25)

axes[1].plot(seismic_norm, t_ms, lw=0.9, label="Seismic", color="black")
axes[1].plot(synthetic, t_ms, lw=0.9, label="Synthetic", color="tab:red", alpha=0.85)
axes[1].set_xlabel("Normalized amplitude")
axes[1].set_title(f"Deterministic LS: corr={metrics['corr']:.3f}, nrmse={metrics['nrmse']:.3f}")
axes[1].legend(loc="best")
axes[1].grid(True, alpha=0.25)

axes[2].plot(seismic_norm - synthetic, t_ms, lw=0.9, color="tab:gray")
axes[2].axvline(0.0, color="black", lw=0.8, alpha=0.5)
axes[2].set_xlabel("Residual")
axes[2].set_title("Residual")
axes[2].grid(True, alpha=0.25)

fig_det_synthetic_path = output_dir / f"qc_03_deterministic_synthetic_vs_seismic_{well_name}.png"
savefig(fig_det_synthetic_path)
plt.show()

In [ ]:
expected_ff, expected_ampl, expected_power, expected_phase = get_spectrum(expected_wavelet, to_degree=True)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
axes[0].plot(expected_wavelet.basis * 1000.0, expected_wavelet.values, lw=1.3, color="tab:blue")
axes[0].axvline(0.0, color="black", lw=0.8, alpha=0.5)
axes[0].set_xlabel("Time (ms)")
axes[0].set_title("compute_expected_wavelet")
axes[0].grid(True, alpha=0.25)

axes[1].plot(expected_ff, expected_ampl, lw=1.2, color="tab:orange")
axes[1].set_xlim(0, min(125, expected_ff[-1]))
axes[1].set_xlabel("Frequency (Hz)")
axes[1].set_title("Amplitude spectrum")
axes[1].grid(True, alpha=0.25)

axes[2].plot(expected_ff, expected_phase, lw=1.0, color="tab:green")
axes[2].set_xlim(0, min(125, expected_ff[-1]))
axes[2].set_xlabel("Frequency (Hz)")
axes[2].set_ylabel("Phase (deg)")
axes[2].set_title("Phase")
axes[2].grid(True, alpha=0.25)

fig_expected_wavelet_path = output_dir / f"qc_04_compute_expected_wavelet_spectrum_{well_name}.png"
savefig(fig_expected_wavelet_path)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 5), sharey=True)
t_expected_ms = twt_expected * 1000.0

axes[0].plot(expected_ref_used, t_expected_ms, lw=0.8, color="tab:purple")
axes[0].invert_yaxis()
axes[0].set_xlabel("Reflectivity")
axes[0].set_ylabel("Relative TWT (ms)")
axes[0].set_title("Reflectivity at 2 ms")
axes[0].grid(True, alpha=0.25)

axes[1].plot(expected_seismic_norm, t_expected_ms, lw=0.9, label="Seismic", color="black")
axes[1].plot(expected_synthetic, t_expected_ms, lw=0.9, label="Synthetic", color="tab:red", alpha=0.85)
axes[1].set_xlabel("Normalized amplitude")
axes[1].set_title(f"compute_expected_wavelet: corr={expected_corr:.3f}, nrmse={expected_nrmse:.3f}")
axes[1].legend(loc="best")
axes[1].grid(True, alpha=0.25)

axes[2].plot(expected_seismic_norm - expected_synthetic, t_expected_ms, lw=0.9, color="tab:gray")
axes[2].axvline(0.0, color="black", lw=0.8, alpha=0.5)
axes[2].set_xlabel("Residual")
axes[2].set_title("Residual")
axes[2].grid(True, alpha=0.25)

fig_expected_synthetic_path = output_dir / f"qc_05_compute_expected_synthetic_vs_seismic_{well_name}.png"
savefig(fig_expected_synthetic_path)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
axes[0].plot(wavelet_t * 1000.0, wavelet_amp / np.max(np.abs(wavelet_amp)), lw=1.2, label="Deterministic LS")
axes[0].plot(expected_wavelet.basis * 1000.0, expected_wavelet.values / np.max(np.abs(expected_wavelet.values)), lw=1.2, label="compute_expected_wavelet")
axes[0].axvline(0.0, color="black", lw=0.8, alpha=0.5)
axes[0].set_xlabel("Time (ms)")
axes[0].set_title("Normalized wavelets")
axes[0].legend(loc="best")
axes[0].grid(True, alpha=0.25)

axes[1].plot(freq_hz, amp_spec / np.max(amp_spec), lw=1.1, label="Deterministic LS")
axes[1].plot(expected_ff, expected_ampl / np.max(expected_ampl), lw=1.1, label="compute_expected_wavelet")
axes[1].set_xlim(0, min(125, expected_ff[-1]))
axes[1].set_xlabel("Frequency (Hz)")
axes[1].set_title("Normalized amplitude spectrum")
axes[1].legend(loc="best")
axes[1].grid(True, alpha=0.25)

fig_overlay_path = output_dir / f"qc_06_wavelet_spectrum_comparison_{well_name}.png"
savefig(fig_overlay_path)
plt.show()

## 输出清单


In [ ]:
figure_paths = [
    fig_depth_time_path,
    fig_det_wavelet_path,
    fig_det_synthetic_path,
    fig_expected_wavelet_path,
    fig_expected_synthetic_path,
    fig_overlay_path,
]
artifact_paths = [
    local_tdt_path,
    matched_path,
    wavelet_path,
    qc_path,
    scan_path,
    expected_wavelet_path,
    expected_input_path,
    expected_synthetic_qc_path,
    summary_path,
]

print("Figures in notebook order:")
for path in figure_paths:
    print(path)

print("\nData artifacts:")
for path in artifact_paths:
    print(path)